# 1. Preparación, entrenamiento y trazas
Ejecutar en Linux con el kernel de `.venv`. Los datos y modelos quedan locales e ignorados por Git. Este notebook llama a los adaptadores del protocolo; los notebooks históricos son solo referencia.


In [1]:
from pathlib import Path
import platform
import sys
import os
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'shared' / 'contract.py').is_file()), None)
assert raiz is not None, f'No se encuentra la raíz del repositorio desde {Path.cwd()}'
os.chdir(raiz)
sys.path.insert(0, str(raiz))
from shared.contract import ROOT, validate_splits
assert Path.cwd().resolve() == ROOT
print(platform.platform(), sys.version.split()[0], ROOT)
assert sys.platform.startswith('linux'), 'Entrenar LSTM en el entorno Linux validado'


Linux-5.15.148-tegra-aarch64-with-glibc2.35 3.10.12 /home/jetson/Desktop/greenhouse-online-anomaly-detection


## Datos físicos congelados
Si faltan los archivos locales, revisar primero su tamaño y procedencia en `data_contract/manifests/imported_files.json`; la hidratación verifica hashes y no modifica la fuente.


In [2]:
# Ejecutar solo si faltan los archivos ignorados y existe el checkout fuente local.
# from shared.data_preparation.hydrate import hydrate
# hydrate(Path('../Clasificaci-n-y-correcci-n'))
contract = validate_splits()
assert [contract['splits'][s]['rows'] for s in ('train', 'validation', 'test')] == [172873, 28800, 31680]
print(contract['data_source'], contract['manifest_id'])


Agroconnect 4cc10ef96d923a326a86c0882a356a4d09d35a6da6553bfdbb60dd80a5f0737c


## LightGBM con noche y contexto interior/exterior
Es un experimento nuevo de 68 variables. El v8 original permanece como referencia de 66 variables.


In [3]:
from approaches.lightgbm.night_context import train as train_lightgbm, export_test as export_lightgbm
lightgbm_run = train_lightgbm()
lightgbm_trace, lightgbm_isolated = export_lightgbm(lightgbm_run)
print(lightgbm_run, lightgbm_trace, lightgbm_isolated, sep='\n')


/home/jetson/Desktop/greenhouse-online-anomaly-detection/experiments/runs/lightgbm_night_v1/20261104T114333Z
/home/jetson/Desktop/greenhouse-online-anomaly-detection/results/traces/lightgbm_night_test.parquet
/home/jetson/Desktop/greenhouse-online-anomaly-detection/results/traces/lightgbm_night_model2_isolated_test.parquet


## LSTM causal
Model 1 y Model 2 se entrenan con train; validation selecciona checkpoints y umbral. Test solo se usa al exportar trazas. Puede tardar bastante.


In [4]:
from approaches.lstm.adapter import train as train_lstm, export_test as export_lstm
lstm_run = train_lstm()
lstm_trace, lstm_isolated = export_lstm(lstm_run)
print(lstm_run, lstm_trace, lstm_isolated, sep='\n')


NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
NvMapMemAllocInternalTagged: 1075072515 error 12
NvMapMemHandleAlloc: error 0
2026-11-04 12:45:27.919281: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 262738080 exceeds 10% of free system memory.
2026-11-04 12:45:28.842583: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 262738080 exceeds 10% of free system memory.


Epoch 1/50
676/676 - 37s - 55ms/step - accuracy: 0.7680 - loss: 0.5101 - pr_auc: 0.5628 - precision: 0.3457 - recall: 0.7164 - roc_auc: 0.8253 - val_accuracy: 0.9181 - val_loss: 0.2433 - val_pr_auc: 0.8404 - val_precision: 0.7800 - val_recall: 0.6985 - val_roc_auc: 0.9249 - learning_rate: 0.0010
Epoch 2/50
676/676 - 34s - 50ms/step - accuracy: 0.9412 - loss: 0.2026 - pr_auc: 0.9270 - precision: 0.7462 - recall: 0.8855 - roc_auc: 0.9708 - val_accuracy: 0.9645 - val_loss: 0.1285 - val_pr_auc: 0.9188 - val_precision: 0.9776 - val_recall: 0.8023 - val_roc_auc: 0.9516 - learning_rate: 0.0010
Epoch 3/50
676/676 - 26s - 39ms/step - accuracy: 0.9732 - loss: 0.1099 - pr_auc: 0.9727 - precision: 0.8778 - recall: 0.9418 - roc_auc: 0.9903 - val_accuracy: 0.9679 - val_loss: 0.1208 - val_pr_auc: 0.9194 - val_precision: 0.9827 - val_recall: 0.8190 - val_roc_auc: 0.9514 - learning_rate: 0.0010
Epoch 4/50
676/676 - 27s - 40ms/step - accuracy: 0.9809 - loss: 0.0792 - pr_auc: 0.9834 - precision: 0.9111 -

2026-11-04 12:51:35.874924: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 285727840 exceeds 10% of free system memory.
2026-11-04 12:51:36.733740: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 285727840 exceeds 10% of free system memory.


Epoch 1/50
96/96 - 10s - 103ms/step - accuracy: 0.4072 - loss: 1.4885 - val_accuracy: 0.5241 - val_loss: 1.3391 - val_macro_f1: 0.4747 - learning_rate: 0.0010
Epoch 2/50
96/96 - 4s - 44ms/step - accuracy: 0.6894 - loss: 0.7998 - val_accuracy: 0.7755 - val_loss: 0.9661 - val_macro_f1: 0.7467 - learning_rate: 0.0010
Epoch 3/50
96/96 - 4s - 45ms/step - accuracy: 0.8686 - loss: 0.3634 - val_accuracy: 0.8114 - val_loss: 0.8799 - val_macro_f1: 0.7738 - learning_rate: 0.0010
Epoch 4/50
96/96 - 4s - 42ms/step - accuracy: 0.9098 - loss: 0.2426 - val_accuracy: 0.8057 - val_loss: 0.9271 - val_macro_f1: 0.7647 - learning_rate: 0.0010
Epoch 5/50
96/96 - 4s - 41ms/step - accuracy: 0.9255 - loss: 0.1994 - val_accuracy: 0.8203 - val_loss: 0.9656 - val_macro_f1: 0.7774 - learning_rate: 0.0010
Epoch 6/50
96/96 - 5s - 54ms/step - accuracy: 0.9354 - loss: 0.1689 - val_accuracy: 0.8197 - val_loss: 1.0507 - val_macro_f1: 0.7725 - learning_rate: 0.0010
Epoch 7/50
96/96 - 5s - 53ms/step - accuracy: 0.9401 - l

Guardar las rutas de ambos runs. El notebook 02 verifica la paridad de objetivos y genera el informe comparativo. No versionar las salidas de este notebook.
